# Task 2.1 — Dataset Selection and Justification

## Dataset: sklearn Wine Dataset (178 samples, 13 features, 3 classes)

**What it is:**  
The Wine dataset contains 178 samples of Italian wines described by 13 real-valued physicochemical features (alcohol, ash, flavanoids, etc.) with 3 cultivar class labels. It is available directly via `sklearn.datasets.load_wine()` with no external download.

**Why it suits multi-class SVM (analogous to USPS in the paper):**  
- Genuine 3-class problem ($Q=3$) matching the MSVMpack evaluation structure.  
- Classes are **well-separated** in feature space — a linear/RBF maximum-margin boundary can cleanly partition them, consistent with the separability assumption in Definition 1.  
- All features are **real-valued continuous** ($\mathcal{X}\subseteq\mathbb{R}^{13}$), exactly the input space assumed in Section 2.  
- Same classification task type as USPS (digit recognition across $Q$ classes from fixed-length real-valued feature vectors).

**Limitations vs USPS / CB513:**  
- **Much smaller** (178 vs 9,298 / 84,119 samples) — MSVMpack's decomposition and memory advantages are not observable.  
- **Lower-dimensional** (13 vs 256 / sequence-length features) — high-dimensional RKHS geometry is less relevant.  
- **No spatial or sequential structure** — image/sequence kernels cannot be demonstrated.

In [1]:
# ── Imports & seed ────────────────────────────────────────────────
import numpy as np, random
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE); random.seed(RANDOM_STATE)

from sklearn.datasets import load_wine
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

# ── Load ────────────────────────────────────────────────────────────
wine = load_wine()
X, y = wine.data, wine.target
print(f"Shape: {X.shape}, Classes: {np.unique(y)}")
print(f"Class counts: {dict(zip(*np.unique(y, return_counts=True)))}")


Shape: (178, 13), Classes: [0 1 2]
Class counts: {np.int64(0): np.int64(59), np.int64(1): np.int64(71), np.int64(2): np.int64(48)}


**Section 2 — Input Space $\mathcal{X}$:** We verify $n=178$, $d=13$, $Q=3$.

In [2]:
# ── Standardise ────────────────────────────────────────────────────
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
print("Feature 0 before:", round(X[:,0].mean(),2), round(X[:,0].std(),2))
print("Feature 0 after :", round(X_scaled[:,0].mean(),6), round(X_scaled[:,0].std(),6))


Feature 0 before: 13.0 0.81
Feature 0 after : 0.0 1.0


**Section 2 — Kernel geometry:** StandardScaler ensures all 13 features contribute equally to the RBF/linear kernel inner product, preventing large-magnitude features from dominating the margin.

In [3]:
# ── 80/20 stratified split ─────────────────────────────────────────
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.20, random_state=RANDOM_STATE, stratify=y)
print(f"Train: {X_train.shape}, Test: {X_test.shape}")
print(f"Train dist: {dict(zip(*np.unique(y_train, return_counts=True)))}")
print(f"Test  dist: {dict(zip(*np.unique(y_test, return_counts=True)))}")


Train: (142, 13), Test: (36, 13)
Train dist: {np.int64(0): np.int64(47), np.int64(1): np.int64(57), np.int64(2): np.int64(38)}
Test  dist: {np.int64(0): np.int64(12), np.int64(1): np.int64(14), np.int64(2): np.int64(10)}


**Table 2 — Evaluation protocol:** Stratified 80/20 split mirrors the train/test evaluation used in the paper. `random_state=42` ensures reproducibility.